# Step02, Task 02: Train `gpp_fTair` Hybrid Model -- Offline training version

Author: Xu Shan

Notebook version of `train_gpp_fTair.jl`.

This workflow trains a neural-network emulator (offline) for GPP from FLUXNET-based forcing data, derives a temperature stress function from the learned GPP sensitivity to `Tair`, saves the checkpoint, and generates the same figures as the script.

The notebook keeps the same numerical logic as the `.jl` version, but it is split into smaller sections so you can inspect intermediate variables, rerun only selected steps, and iterate on choices such as site splits, feature lists, architecture, and the transformation from sensitivity to stress.

# Motivation

Different process based models (PBMs) have different representations of the temperature response of GPP, and they can differ substantially from each other and from observations. For example, CASA treats GPP-Temperature sensitivity curves differently when temperature is below optimum temperature and above the optimum temperature. In contrast, Horn et al treates the sensitivity curves symmetrically around the optimum temperature. The goal of this workflow is to derive a data-driven temperature stress function that can be used in the PBM to replace the original temperature response curve, and to evaluate how well the learned stress function can reproduce observed GPP.

# Outputs

The main outputs are:

- a trained Flux checkpoint `gpp_model.jld2`,
- a normalized temperature stress curve `f_Tair`, and
- diagnostic figures for model fit and temperature-response behavior.

In [ ]:
using Pkg
Pkg.add(["Flux", "ProgressMeter", "Zarr", "Statistics", "JLD2", "ForwardDiff", "CairoMakie"])

using Flux, Statistics, ProgressMeter
using Zarr
using JLD2
using ForwardDiff
using CairoMakie

project_root = isdir(joinpath(pwd(), "tutorials")) ? pwd() : abspath(joinpath(pwd(), "..", ".."))
hybrid_dir = joinpath(project_root, "tutorials", "hybrid_inversion")
save_path = joinpath(project_root, "gpp_model.jld2")

## Configuration

What shall be the inputs of the model? What is the target variable? Which sites are used for training and which for validation? Which feature corresponds to `Tair`?
- Meteorogolical variables determine the temperature response of GPP, which variables shall we include as predictors?
- Shall we consider soil moisture and VPD as predictors, or do we want to isolate the temperature response by excluding them?
- Static variables such as plant functional type (PFT) and mean annual temperature (MAT) can help the model learn different temperature responses for different vegetation types and climates, but they are not available at the same temporal resolution as the target variable. Do we want to include them as predictors?
- What else...?

Technically...

This section defines the remote Zarr source, the predictor variables, the target variable, and the QC mask.

Important details:

- `input_list` fixes the exact feature order used during training and later during inference.
- `tair_idx` identifies the air-temperature feature whose sensitivity will be analyzed with `ForwardDiff`.
- `training_sites` and `validation_sites` implement a simple site-based split that you can change if needed.

The notebook also uses an explicit `project_root` so the saved checkpoint path is stable regardless of whether Jupyter is launched from the repository root or from `tutorials/hybrid_inversion`.

In [ ]:
data_path = "https://s3.bgc-jena.mpg.de:9000/sindbad/FLUXNET_v2023_12_1D_REPLACED_Noise003_v1.zarr"

input_list = [
    "atmCO2_SCRIPPS_global",
    "SW_IN_ERAIv2_gfld",
    "P_ERAIv2_gfld",
    "SW_IN_POT_ONEFlux",
    "NETRAD_ERAIv2_gfld",
    "TA_DayTime_ERAIv2_gfld",
    "VPD_DayTime_ERAIv2_gfld",
]
output_list = ["GPP_NT"]
output_mask_list = ["GPP_QC_NT_merged"]
tair_idx = findfirst(==("TA_DayTime_ERAIv2_gfld"), input_list)

training_sites = collect(1:164)
validation_sites = collect(165:205)

## Load Data

Here the notebook opens the remote Zarr store and loads the forcing variables, target GPP, and QC mask into dense Julia arrays.

The predictors are stacked into an array with dimensions `T × S × F`, where `T` is time, `S` is site, and `F` is feature. This makes the later conversion into training samples straightforward while preserving the original indexing for diagnostics.

In [ ]:
println("Opening remote Zarr store...")
sindbad_data = zopen(data_path)

println("Loading input variables $(input_list)...")
X_raw = cat([Float32.(sindbad_data[v][:, :]) for v in input_list]...; dims=3)

println("Loading target: $(output_list[1])...")
Y_raw = Float32.(sindbad_data[output_list[1]][:, :])

println("Loading QC mask: $(output_mask_list[1])...")
QC_raw = Float32.(sindbad_data[output_mask_list[1]][:, :])

T, S, F = size(X_raw)
println("Data shape: T=$T timesteps × S=$S sites × F=$F features")

## Build Training and Validation Samples

The helper below converts the gridded arrays into supervised-learning samples.

A sample is retained only when:

- the target GPP value is not `NaN`,
- the QC mask is available and above `0.85`,
- none of the input features are `NaN`, and
- daytime air temperature is above freezing.

That final filter is intentional in this script: it restricts the learned response to the active-temperature regime before the stress function is derived.

In [ ]:
function collect_samples(sites)
    xs = Vector{Vector{Float32}}()
    ys = Vector{Float32}()
    for s in sites
        for t in 1:T
            y = Y_raw[t, s]
            qc = QC_raw[t, s]
            x = X_raw[t, s, :]
            isnan(y) && continue
            isnan(qc) && continue
            qc < 0.85f0 && continue
            any(isnan, x) && continue
            x[tair_idx] < 0f0 && continue
            push!(xs, x)
            push!(ys, y)
        end
    end
    X = reduce(hcat, xs)
    Y = reshape(ys, 1, :)
    return X, Y
end

println("Collecting training samples ($(length(training_sites)) sites)...")
X_train, Y_train = collect_samples(training_sites)

println("Collecting validation samples ($(length(validation_sites)) sites)...")
X_val, Y_val = collect_samples(validation_sites)

println("Training   samples : $(size(X_train, 2))")
println("Validation samples : $(size(X_val, 2))")

## Normalize and Train

- How shall we treat the predictors and the target variables? are they normally distributed? Do they have outliers? Do we want to transform them (e.g., log-transform GPP) or just z-score them?
- Do we need to consider the temporal autocorrelation in the data when we split training and validation sets, or when we shuffle the samples during training?
- What architecture shall we use for the neural network? How many hidden layers and how many neurons in each layer? Which activation function? Which optimizer and learning rate?
- Which machine learning algorithms better fits for GPP-Temperature sensitivity? Do we want to try other algorithms such as random forest or gradient boosting? Do we need to consider the interpretability of the model when we choose the algorithm?
- Shall we try sequential learning algorithms like RNN, transformer, or temporal convolutional networks that can explicitly capture the temporal dynamics of GPP and its response to temperature, instead of a simple feed-forward network?
- If so, how shall we consider the temporal lagging effects of temperature to GPP response? Do we want to include lagged temperature predictors, or do we want the model to learn the lagged response implicitly from the data by using sequential learning algorithms?

Small examples start here...

Inputs and outputs are z-scored using statistics computed from the training data only.

This serves two purposes:

- it makes optimization with Adam much more stable, and
- it gives you normalization constants that can later be reused when the trained model is loaded into Sindbad.

The current network is a simple feed-forward model with two hidden layers. Loss curves for both training and validation are stored so you can quickly assess whether the fit is stable or drifting toward overfitting.

In [ ]:
μ_x = mean(X_train; dims=2)
σ_x = std(X_train; dims=2) .+ Float32(1e-6)

X_train_n = (X_train .- μ_x) ./ σ_x
X_val_n = (X_val .- μ_x) ./ σ_x

μ_y = mean(Y_train)
σ_y = std(Y_train) + Float32(1e-6)

Y_train_n = (Y_train .- μ_y) ./ σ_y
Y_val_n = (Y_val .- μ_y) ./ σ_y

batchsize = 512
loader = Flux.DataLoader((X_train_n, Y_train_n); batchsize=batchsize, shuffle=true)

model = Chain(
    Dense(F => 64, relu),
    Dense(64 => 32, relu),
    Dense(32 => 1),
)

println(model)
opt_state = Flux.setup(Flux.Adam(1e-3), model)

n_epochs = 50
train_loss = Float32[]
val_loss = Float32[]

println("Training for $n_epochs epochs...")
@showprogress "Epoch " for epoch in 1:n_epochs
    epoch_loss = 0f0
    n_batches = 0
    for (x_b, y_b) in loader
        loss, grads = Flux.withgradient(model) do m
            Flux.mse(m(x_b), y_b)
        end
        Flux.update!(opt_state, model, grads[1])
        epoch_loss += loss
        n_batches += 1
    end
    push!(train_loss, epoch_loss / n_batches)
    push!(val_loss, Flux.mse(model(X_val_n), Y_val_n))
end

## Evaluation

After training, predictions are transformed back to the original GPP units and summarized with a validation-set `R²`.

This is only a compact summary metric, but it is useful as a first sanity check before deriving the temperature-stress curve from the trained emulator.

In [ ]:
Y_pred = model(X_val_n) .* σ_y .+ μ_y

ss_res = sum((Y_val .- Y_pred) .^ 2)
ss_tot = sum((Y_val .- mean(Y_val)) .^ 2)
r2 = 1f0 - ss_res / ss_tot

println("Final train loss (normalised MSE) : $(round(train_loss[end]; digits=4))")
println("Final val   loss (normalised MSE) : $(round(val_loss[end]; digits=4))")
println("Validation R²                     : $(round(r2; digits=4))")

## Derive `f_Tair` from ForwardDiff Sensitivity

- Fun part comes here! We have a trained GPP emulator, and now we want to derive a temperature stress function from the sensitivity of the emulator to air temperature. How?
- Deep thinking...does the success of the training on GPP guarantee that the learned temperature response is meaningful? How can we evaluate the quality of the learned temperature response? Do we want to compare it with observed temperature responses, or with the original PBM temperature response curve, or with other data-driven estimates of temperature response?

This is the main modeling step.

Rather than fitting a separate analytical temperature-stress function, the notebook derives `f_Tair` from the sensitivity of the trained GPP emulator to air temperature.

The idea is:

1. hold all normalized predictors at their training mean,
2. sweep temperature across the observed training range,
3. compute `∂GPP/∂Tair` with `ForwardDiff`,
4. take the absolute sensitivity magnitude, and
5. transform that magnitude into a normalized stress curve.

In this implementation:

- `1` means no stress,
- `0` means full stress,
- the optimum temperature is where the absolute slope is smallest.

In [ ]:
x_mean_n = zeros(Float32, F)

tair_obs_flat = filter(!isnan, vec(X_raw[:, training_sites, tair_idx]))
tair_range_raw = collect(Float32, range(max(0f0, minimum(tair_obs_flat)), maximum(tair_obs_flat); length=200))
tair_range_n = (tair_range_raw .- μ_x[tair_idx]) ./ σ_x[tair_idx]

function gpp_vs_tair(tair_n::T) where T<:Real
    x = Vector{T}(x_mean_n)
    x[tair_idx] = tair_n
    return only(model(reshape(x, F, 1)))
end

sens_sweep = [ForwardDiff.derivative(gpp_vs_tair, t) for t in tair_range_n]
abs_sens = abs.(sens_sweep)
abs_sens_lo, abs_sens_hi = extrema(abs_sens)
abs_sens_norm = (abs_sens .- abs_sens_lo) ./ (abs_sens_hi - abs_sens_lo + Float32(1e-6))
stress_function = 1f0 .- abs_sens_norm

t_opt = tair_range_raw[argmax(stress_function)]
t_lo = tair_range_raw[argmin(stress_function)]
println("Tair range   : [$(round(minimum(tair_range_raw); digits=1)), $(round(maximum(tair_range_raw); digits=1))] °C")
println("Optimal T    : $(round(t_opt; digits=1)) °C  (stress = 1.0)")
println("Most-stressed: $(round(t_lo; digits=1)) °C  (stress = 0.0)")

## Save Checkpoint

The saved checkpoint includes everything needed for downstream reuse inside Sindbad:

- trained model weights,
- input/output normalization statistics,
- feature order,
- training history, and
- the derived temperature-stress quantities.

Keeping all of that in one `JLD2` file makes the later `precompute` step in `gppAirT_externalNN` much simpler.

In [ ]:
jldsave(save_path;
    model_state=Flux.state(model),
    μ_x, σ_x,
    μ_y, σ_y,
    input_list,
    train_loss, val_loss,
    tair_range_raw,
    sens_sweep,
    abs_sens_norm,
    stress_function,
    abs_sens_lo=abs_sens_lo,
    abs_sens_hi=abs_sens_hi,
)

println("Model + stress function saved to: $save_path")

## Validation Figures

The final section creates a few quick-look diagnostics:

- the derived `f_Tair` curve,
- training and validation loss curves,
- a time-sorted observed-versus-predicted comparison, and
- a binned `Tair` versus GPP relationship.

Together these plots help you judge whether the learned response is numerically stable and whether the derived temperature sensitivity is physically plausible enough to export to Sindbad.

In [ ]:
function collect_samples_with_tair(sites)
    xs = Vector{Vector{Float32}}()
    ys = Vector{Float32}()
    ts = Vector{Float32}()
    tidx = Vector{Int}()
    for s in sites
        for t in 1:T
            y = Y_raw[t, s]
            qc = QC_raw[t, s]
            x = X_raw[t, s, :]
            (isnan(y) || isnan(qc) || qc < 0.85f0 || any(isnan, x)) && continue
            x[tair_idx] < 0f0 && continue
            push!(xs, x)
            push!(ys, y)
            push!(ts, x[tair_idx])
            push!(tidx, t)
        end
    end
    X = reduce(hcat, xs)
    Y = reshape(ys, 1, :)
    return X, Y, ts, tidx
end

X_val2, Y_val2, Tair_val, Ti_val = collect_samples_with_tair(validation_sites)
X_val2_n = (X_val2 .- μ_x) ./ σ_x
Y_pred2 = vec(model(X_val2_n) .* σ_y .+ μ_y)
Y_obs2 = vec(Y_val2)

In [ ]:
fig1 = Figure(size=(700, 400))
ax1 = Axis(fig1[1, 1]; xlabel="Air Temperature (°C)", ylabel="f_Tair (stress function)", title="Tair temperature response")
lines!(ax1, tair_range_raw, stress_function; color=:firebrick, linewidth=2)
vlines!(ax1, [t_opt]; color=:gray, linestyle=:dash, label="T_opt = $(round(t_opt; digits=1)) °C")
axislegend(ax1; position=:lb)
save(joinpath(hybrid_dir, "fig1_stress_function.png"), fig1)

fig2 = Figure(size=(700, 400))
ax2 = Axis(fig2[1, 1]; xlabel="Epoch", ylabel="MSE loss (normalised)", title="Training and validation loss")
lines!(ax2, 1:n_epochs, train_loss; color=:steelblue, linewidth=2, label="train")
lines!(ax2, 1:n_epochs, val_loss; color=:darkorange, linewidth=2, label="validation")
axislegend(ax2; position=:rt)
save(joinpath(hybrid_dir, "fig2_loss_curves.png"), fig2)

In [ ]:
ordv = sortperm(Ti_val)
yo_srt = Y_obs2[ordv]
yp_srt = Y_pred2[ordv]

fig3 = Figure(size=(900, 400))
ax3 = Axis(fig3[1, 1]; xlabel="Sample index (time-sorted)", ylabel="GPP (gC m⁻² d⁻¹)", title="Observed vs estimated GPP")
lines!(ax3, 1:length(yo_srt), yo_srt; color=(:steelblue, 0.7), linewidth=1, label="Observed")
lines!(ax3, 1:length(yp_srt), yp_srt; color=(:firebrick, 0.7), linewidth=1, label="Estimated")
axislegend(ax3; position=:lt)
save(joinpath(hybrid_dir, "fig3_timeseries.png"), fig3)

nbins = 60
edges = range(minimum(Tair_val), maximum(Tair_val); length=nbins + 1)
bin_obs_mean = Float32[]
bin_pred_mean = Float32[]
bin_centers = Float32[]
for i in 1:nbins
    lo, hi = edges[i], edges[i + 1]
    mask = (Tair_val .>= lo) .& (Tair_val .< hi)
    sum(mask) < 3 && continue
    push!(bin_centers, (lo + hi) / 2)
    push!(bin_obs_mean, mean(Y_obs2[mask]))
    push!(bin_pred_mean, mean(Y_pred2[mask]))
end

fig4 = Figure(size=(700, 500))
ax4 = Axis(fig4[1, 1]; xlabel="Air Temperature (°C)", ylabel="GPP (gC m⁻² d⁻¹)", title="GPP vs Tair")
scatter!(ax4, Tair_val, Y_obs2; color=(:steelblue, 0.15), markersize=3, label="Observed (raw)")
scatter!(ax4, Tair_val, Y_pred2; color=(:firebrick, 0.15), markersize=3, label="Estimated (raw)")
lines!(ax4, bin_centers, bin_obs_mean; color=:steelblue, linewidth=2.5, label="Observed (binned mean)")
lines!(ax4, bin_centers, bin_pred_mean; color=:firebrick, linewidth=2.5, label="Estimated (binned mean)")
axislegend(ax4; position=:lt, merge=true)
save(joinpath(hybrid_dir, "fig4_scatter_tair_gpp.png"), fig4)

## Outputs

This notebook writes two main kinds of artifacts:

- Checkpoint: `gpp_model.jld2` saved in the repository root
- Figures: saved in `tutorials/hybrid_inversion/`

Because the checkpoint is written to the repository root, the downstream sensitivity scripts and notebooks can load it from a stable location without depending on the notebook launch directory.

Natural follow-up improvements you might want to explore later:

- alternative site splits or cross-validation,
- a smoother or more constrained architecture for the temperature response,
- different transformations from sensitivity to `f_Tair`, and
- additional validation metrics beyond `R²`.

## Any ideas to improve the training and the stress function?...